## **Introduction to MFA (Work in progress...)**

This notebook provides a methodological illustration of how **fragmented public datasets can be fused into a coherent material flow picture**. The outputs are **not market estimates** but **proof-of-concept reconstructions** showing what becomes possible when administrative data silos are integrated.

The purpose is to demonstrate how **secondary raw material stocks and flows**—waste streams, by-products, and end-of-life goods—can be mapped with **much higher physical and geographic resolution** than is typically available in published statistics.

> To the best of our knowledge, **no publicly available online source currently provides this type of facility-, material-, and flow-resolved view of secondary raw materials**. What exists today are either high-level national aggregates or isolated datasets that do not connect generation, treatment, capacity, and cross-border movement in one system.

---

## **Why this matters**

Secondary raw materials are central to supply security and circular economy strategies. Yet decision-relevant physical data—*how much material exists, where it is, and where it flows*—is usually:

- **Siloed** across statistical, environmental, and customs authorities  
- **Aggregated** beyond what firms or planners can use  
- **Structurally incomplete**, lacking links between waste types, treatment routes, and industrial activities  

As a result, even though large volumes of data exist, **they cannot normally answer questions like**:
- *Which regions are hotspots for recoverable copper-bearing waste?*  
- *Which industries are the main sources?*  
-   *Which actors are the major contributors?* (requires Orbis-level data not currently available)
- *Where does it get shipped and treated?*
- *Where's the market for potential secondary raw materials?*

This notebook shows how those questions can be approached by **re-engineering the data into a single material flow system**.

---

## **Data sources used**

Individually, the datasets below are partial. Combined, they allow reconstruction of **stocks, flows, and processing capacity of secondary materials across Europe**.

| Source | What it contributes |
|-------|---------------------|
| **Waste generation & treatment (Eurostat)** | Physical quantities of waste by country, NACE activity, and EWC-Stat code |
| **Waste shipments** | Cross-border flows by waste type (LoW codes) and treatment operation |
| **Waste facilities** | Regional treatment capacity (NUTS-2 level) |
| **Structural Business Statistics (SBS)** | Economic structure used for **allocating waste to regional hotspots and actors** |
| **Trade in secondary raw materials** | Physical trade of recyclable commodities |
| **Material Flow Accounts (MFA)** | System-wide mass balance (extraction, imports, exports, consumption) |
| **Material dependency indicators** | Strategic context for critical and secondary materials |

The novelty is **not the data itself**—most of it is public—but the **integration**:  
linking waste types to industries, to treatment routes, to regions, and to cross-border flows within one coherent physical accounting framework.

In [4]:
import os
from pathlib import Path
os.chdir(Path().resolve().parent)
import pandas as pd
import numpy as np

In [5]:
from src.io_file import load_dataset,load_csv
from src.utils.smart_format import smart_format
from src.features import extend_eurostat_dataset
%load_ext autoreload
%autoreload 2
pd.options.display.float_format = smart_format
pd.options.display.max_columns = 50
pd.options.display.max_rows = 20


## Waste generation by country, Nace rev2 activity and waste type (EWC-Stat 4 codes)

All EWC-Stat 4 waste codes included in the dataset:

In [6]:
wasgen = extend_eurostat_dataset(load_dataset("env_wasgen"),['nace_r2','waste','geo'])
wasgen['waste_description'].unique()

array(['Primary waste (TOTAL minus SEC)',
       'Secondary waste (W033+W103+W128_13)', 'Total waste',
       'Waste excluding major mineral wastes',
       'Chemical and medical wastes (subtotal)', 'Spent solvents',
       'Acid, alkaline or saline wastes', 'Used oils', 'Chemical wastes',
       'Industrial effluent sludges',
       'Sludges and liquid wastes from waste treatment',
       'Health care and biological wastes',
       'Metallic wastes (W061+W062+W063)', 'Metal wastes, ferrous',
       'Metal wastes, non-ferrous',
       'Metal wastes, mixed ferrous and non-ferrous',
       'Recyclable wastes (subtotal, W06+W07 except W077)',
       'Glass wastes', 'Paper and cardboard wastes', 'Rubber wastes',
       'Plastic wastes', 'Wood wastes', 'Textile wastes',
       'Waste containing PCB',
       'Equipment (subtotal, W077+W08A+W081+W0841)', 'Discarded vehicles',
       'Batteries and accumulators wastes',
       'Discarded equipment (except discarded vehicles and batteries and a

#### Example: Generated waste for Metal manufacturing industries (C24,C25)

- Top 10 records, annual mean in EU per waste code and country:

In [8]:
pd.read_csv('data/processed/Waste generation/Generated_waste_per_nace_C24_C25_country.csv').head(10)

,country,nace_r2,nace_r2_activity,waste,waste_description,2004,2006,2008,2010,2012,2014,2016,2018,2020,2022,mean_wasgen,std_wasgen
0,Poland,C24_C25,Manufacture of basic metals and fabricated met...,W12-13,Mineral and solidified wastes (subtotal),79.30M,75.38M,71.89M,0.00,0.00,0.00,0.00,0.00,0.00,0.00,75.52M,3.72M
1,Poland,C24_C25,Manufacture of basic metals and fabricated met...,W12_X_127NH,Mineral waste (except non-hazardous dredging s...,79.30M,75.38M,71.89M,0.00,0.00,0.00,0.00,0.00,0.00,0.00,75.52M,3.72M
2,Poland,C24_C25,Manufacture of basic metals and fabricated met...,W12A,"Mineral wastes (except combustion wastes, cont...",67.81M,64.46M,60.66M,0.00,0.00,0.00,0.00,0.00,0.00,0.00,64.31M,3.58M
3,Poland,C24_C25,Manufacture of basic metals and fabricated met...,TOTAL,Total waste,83.26M,80.27M,74.98M,18.05M,21.59M,21.82M,20.58M,23.68M,19.01M,18.66M,38.19M,28.73M
4,Italy,C24_C25,Manufacture of basic metals and fabricated met...,TOTAL,Total waste,24.09M,23.88M,29.80M,20.36M,21.23M,20.37M,21.59M,21.60M,21.58M,22.00M,22.65M,3.01M
5,Germany,C24_C25,Manufacture of basic metals and fabricated met...,TOTAL,Total waste,26.97M,28.15M,26.93M,23.85M,19.20M,21.54M,19.16M,18.62M,18.90M,17.61M,22.09M,4.16M
6,Germany,C24_C25,Manufacture of basic metals and fabricated met...,W12-13,Mineral and solidified wastes (subtotal),22.18M,22.51M,21.21M,0.00,0.00,0.00,0.00,0.00,0.00,0.00,21.97M,"915,000"
7,Germany,C24_C25,Manufacture of basic metals and fabricated met...,W12_X_127NH,Mineral waste (except non-hazardous dredging s...,22.13M,22.43M,21.07M,0.00,0.00,0.00,0.00,0.00,0.00,0.00,21.88M,"914,000"
8,Italy,C24_C25,Manufacture of basic metals and fabricated met...,PRIM,Primary waste (TOTAL minus SEC),0.00,0.00,0.00,20.09M,20.89M,20.03M,21.20M,21.12M,21.16M,21.43M,20.84M,"636,000"
9,Italy,C24_C25,Manufacture of basic metals and fabricated met...,TOT_X_MIN,Waste excluding major mineral wastes,21.84M,22.62M,24.97M,18.07M,19.48M,18.68M,19.93M,20.11M,20.23M,20.55M,20.76M,2.06M


### Find all metal wastes in Poland, C24, C25 activities

- Annual mean generation (tonnes)

In [80]:

c24_c25 = wasgen[wasgen['nace_r2']=='C24_C25']
c24_c25[(c24_c25['geo_description']=='Poland')&
        (c24_c25['unit']=='T')&
        (c24_c25['waste_description'].isin(['Metal wastes, ferrous',
       'Metal wastes, non-ferrous',
       'Metal wastes, mixed ferrous and non-ferrous',
       'Recyclable wastes (subtotal, W06+W07 except W077)',
       'Metallic wastes (W061+W062+W063)',
       'Other mineral wastes (W122+W123+W125)',
       'Mineral and solidified wastes (subtotal)']))].groupby(['geo_description','nace_r2_description','waste','waste_description'])[c24_c25.columns[9:]].sum().mean(axis=1).reset_index().sort_values(by=0,ascending=False)


,geo_description,nace_r2_description,waste,waste_description,0
5,Poland,Manufacture of basic metals and fabricated met...,W12-13,Mineral and solidified wastes (subtotal),22.66M
6,Poland,Manufacture of basic metals and fabricated met...,W12B,Other mineral wastes (W122+W123+W125),2.35M
1,Poland,Manufacture of basic metals and fabricated met...,W061,"Metal wastes, ferrous",2.32M
4,Poland,Manufacture of basic metals and fabricated met...,W06_07A,"Recyclable wastes (subtotal, W06+W07 except W077)","616,440"
0,Poland,Manufacture of basic metals and fabricated met...,W06,Metallic wastes (W061+W062+W063),"603,925"
2,Poland,Manufacture of basic metals and fabricated met...,W062,"Metal wastes, non-ferrous","214,233"
3,Poland,Manufacture of basic metals and fabricated met...,W063,"Metal wastes, mixed ferrous and non-ferrous","16,931"


### SBS for NACE 2 activity and NUTS2 regions

Enables economic allocation of the generated waste to NUTS2 regions per NACE 2 activity.

- Identifies regional hotspots of waste generation

In [8]:
sbs_r_nuts06_r2 = extend_eurostat_dataset(load_dataset(
    "sbs_r_nuts06_r2"
) ,['nace_r2','indic_sb','geo'])

### Example: Find the top NUTS2 regions for Metal manufacturing in poland

- Use local units (number of facilities), persons employed or both as proxy for waste generation.

- (Could also use Orbis data for economic turnover as proxy etc.)

-> Possible to allocate generated waste to regional hotspots.

- Use Orbis data to find biggest producers within that hotspot (Or just use country-specific Orbis data and allocate per company)

In [64]:
sbs_r_nuts06_r2[(sbs_r_nuts06_r2['geo'].str.startswith('PL'))&
                ((sbs_r_nuts06_r2['nace_r2']=='C24')|
                 (sbs_r_nuts06_r2['nace_r2']=='C25'))].sort_values(by='2015',ascending=False).head(10)

,freq,nace_r2,nace_r2_description,indic_sb,indic_sb_description,geo,geo_description,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
56247,A,C25,"Manufacture of fabricated metal products, exce...",V16110,Persons employed - number,PL,Poland,276056.0,272608.0,268048.0,276820.0,279982.0,280143.0,291420.0,301566.0,324957.0,331615.0,365947.0,NaN,NaN
56251,A,C25,"Manufacture of fabricated metal products, exce...",V16110,Persons employed - number,PL2,Makroregion południowy,66917.0,66679.0,65279.0,68530.0,69082.0,68245.0,68074.0,72004.0,77125.0,79369.0,87120.0,89308.0,89269.0
53594,A,C24,Manufacture of basic metals,V16110,Persons employed - number,PL,Poland,74131.0,69682.0,66125.0,65566.0,66436.0,64757.0,64241.0,65693.0,68651.0,70681.0,71335.0,71753.0,70645.0
56259,A,C25,"Manufacture of fabricated metal products, exce...",V16110,Persons employed - number,PL4,Makroregion północno-zachodni,49213.0,48875.0,46166.0,49900.0,50576.0,51376.0,54291.0,55567.0,59774.0,61864.0,68462.0,70405.0,68780.0
56266,A,C25,"Manufacture of fabricated metal products, exce...",V16110,Persons employed - number,PL6,Makroregion północny,47694.0,46744.0,45140.0,46397.0,48722.0,50389.0,52349.0,54500.0,58879.0,57108.0,63265.0,65206.0,65757.0
56253,A,C25,"Manufacture of fabricated metal products, exce...",V16110,Persons employed - number,PL22,Śląskie,45414.0,44723.0,44056.0,46050.0,47377.0,46053.0,45497.0,48084.0,50848.0,52263.0,56190.0,57766.0,56674.0
56248,A,C25,"Manufacture of fabricated metal products, exce...",V16110,Persons employed - number,PL1,Region Centralny (NUTS 2013),43046.0,42577.0,41259.0,38951.0,39450.0,40780.0,43354.0,43609.0,NaN,NaN,NaN,NaN,NaN
56254,A,C25,"Manufacture of fabricated metal products, exce...",V16110,Persons employed - number,PL3,Region Wschodni (NUTS 2013),38043.0,38472.0,38395.0,39626.0,39684.0,38174.0,41300.0,43384.0,NaN,NaN,NaN,NaN,NaN
53598,A,C24,Manufacture of basic metals,V16110,Persons employed - number,PL2,Makroregion południowy,41141.0,38883.0,35374.0,36095.0,36154.0,35462.0,34646.0,34499.0,35307.0,36875.0,35147.0,36234.0,34706.0
55179,A,C25,"Manufacture of fabricated metal products, exce...",V11210,Local units - number,PL,Poland,30236.0,29118.0,28441.0,30263.0,30012.0,29128.0,31339.0,34018.0,36586.0,38249.0,45082.0,46976.0,48428.0


Available economic parameters:

In [41]:
sbs_r_nuts06_r2['indic_sb_description'].unique()

array(['Local units - number', 'Wages and Salaries - million euro',
       'Persons employed - number',
       'Growth rate of employment - percentage',
       'Share of employment in manufacturing total - percentage'],
      dtype=object)

## Waste treatment by country, operations and waste type (EWC Stat 4 codes)

Country level data on how the generated waste is treated.

- Look at the disposal rate of the generated waste in the country of the hotspot (NUTS 2 regional detail not available)  

- See whether the amount of treated waste sums up to generated wastes (Is there under-reporting for treated wastes).

- Likely that large amounts underreporting exist because many by-products aren't 'legally treated'.



### Example: Poland, metal wastes 

In [10]:
wastrt = extend_eurostat_dataset(load_dataset("env_wastrt"),['unit','wst_oper','waste','geo'])

poland_metal_trt = wastrt[wastrt['waste'].isin(['W061','W062','W063'])&
        (wastrt['geo_description']=='Poland')&
        (wastrt['unit']=='T')]#.sort_values(by='mean_ship',ascending=False)


poland_metal_trt.groupby(['waste_description','wst_oper_description'])[['2004', '2006', '2008',
       '2010', '2012', '2014', '2016', '2018', '2020', '2022']].sum().mean(axis=1).reset_index().sort_values(by=0,ascending=False).head(10)



,waste_description,wst_oper_description,0
8,"Metal wastes, ferrous",Waste treatment,8.38M
7,"Metal wastes, ferrous",Recovery - recycling and backfilling (R2-R11),8.38M
6,"Metal wastes, ferrous",Recovery - recycling,8.37M
26,"Metal wastes, non-ferrous",Waste treatment,"840,878"
25,"Metal wastes, non-ferrous",Recovery - recycling and backfilling (R2-R11),"840,384"
24,"Metal wastes, non-ferrous",Recovery - recycling,"840,384"
17,"Metal wastes, mixed ferrous and non-ferrous",Waste treatment,"164,078"
16,"Metal wastes, mixed ferrous and non-ferrous",Recovery - recycling and backfilling (R2-R11),"164,070"
15,"Metal wastes, mixed ferrous and non-ferrous",Recovery - recycling,"164,070"
4,"Metal wastes, ferrous",Recovery - backfilling,"13,284"


- Discrepency between generated treated waste (Actually more treated waste than generated).
- -> Waste products from old landfills has perhaps been recovered, and entered into treatment statistics? (Or the waste is generated in other nace sectors)

### Trade in recyclable raw materials

- How much 'waste' is being exported as secondary raw materials

In [39]:
#trdrrm = extend_eurostat_dataset(load_dataset("env_trdrrm"),['stk_flow','rawmat','partner','unit','geo'])
rawmat = ['Mineral', 'Metal', 'Metal - ferrous',
       'Metal - non ferrous',
       'Metal - non ferrous - copper, aluminium and nickel',
       'Metal - non ferrous - other',
       'Metal - non ferrous - precious metals']


trdrrm['rawmat_description'].unique()
trdrrm[(trdrrm['geo']=='PL')&
       (trdrrm['rawmat_description'].isin(rawmat))&
       #(trdrrm['stk_flow']=='EXP')&
       (trdrrm['unit']=='T')].groupby(['stk_flow_description'])[trdrrm.columns[11:]].sum().mean(axis=1).reset_index().sort_values(by=0,ascending=False)


,stk_flow_description,0
0,Exports,5.70M
1,Imports,2.74M


=> Poland is a net exporter of metallic secondary raw materials

In [134]:
trdrrm[(trdrrm['geo']=='PL')&
       (trdrrm['rawmat_description'].isin(rawmat))&
       (trdrrm['stk_flow']=='EXP')&
       (trdrrm['unit']=='T')].groupby(['stk_flow_description','rawmat_description'])[trdrrm.columns[11:]].sum().mean(axis=1).reset_index().sort_values(by=0,ascending=False)

,stk_flow_description,rawmat_description,0
0,Exports,Metal,2.56M
1,Exports,Metal - ferrous,2.24M
2,Exports,Metal - non ferrous,"321,497"
3,Exports,"Metal - non ferrous - copper, aluminium and ni...","294,155"
6,Exports,Mineral,"254,806"
4,Exports,Metal - non ferrous - other,"24,053"
5,Exports,Metal - non ferrous - precious metals,"3,289"


=> Mostly ferrous raw materials are being exported

In [43]:
trdrrm[(trdrrm['geo']=='PL')&
       (trdrrm['rawmat_description'].isin(rawmat))&
       (trdrrm['stk_flow']=='EXP')&
       (trdrrm['unit']=='T')].groupby(['stk_flow_description','rawmat_description','partner_description'])[trdrrm.columns[11:]].sum().mean(axis=1).reset_index().sort_values(by=0,ascending=False).head(20)


,stk_flow_description,rawmat_description,partner_description,0
28,Exports,Metal,Intra-EU27 (from 2020),1.67M
93,Exports,Metal - ferrous,Intra-EU27 (from 2020),1.41M
20,Exports,Metal,Extra-EU27 (from 2020),"444,377"
86,Exports,Metal - ferrous,Extra-EU27 (from 2020),"414,425"
144,Exports,Metal - non ferrous,Intra-EU27 (from 2020),"261,595"
194,Exports,"Metal - non ferrous - copper, aluminium and ni...",Intra-EU27 (from 2020),"243,471"
292,Exports,Mineral,Intra-EU27 (from 2020),"207,925"
63,Exports,Metal,Türkiye,"155,940"
120,Exports,Metal - ferrous,Türkiye,"155,658"
26,Exports,Metal,India,"124,239"


=> Mostly within EU (Would be nice to find where in the EU it's going)

## Waste facilities per NUTS2 region

This could also be complemented with IED Installations data, and OECD Pre-consented waste details etc. 

- Shows if there's local infrastructure for treating waste locally.

In [12]:
wasfac = extend_eurostat_dataset(load_dataset("env_wasfac"),['freq','indic_env','wst_oper','geo'])

### Example: Poland (NUTS2 region: Makroregion południowy, Major C25 region)

In [16]:
wasfac[wasfac['geo_description']=='Makroregion południowy'].head()
wasfac[(wasfac['geo_description']=='Makroregion południowy')].groupby(['geo_description',
                                                                       'indic_env_description',
                                                                       'wst_oper_description'
                                                                       ])[wasfac.columns[8:]].sum().mean(axis=1).reset_index().sort_values(by=0,ascending=False).head(20)


,geo_description,indic_env_description,wst_oper_description,0
13,Makroregion południowy,Rest capacity - cubic metres,"Disposal - landfill (D1, D5, D12)",17.43M
4,Makroregion południowy,Capacity - tonnes per year,Recovery - recycling and backfilling (R2-R11),4.40M
3,Makroregion południowy,Capacity - tonnes per year,Recovery - energy recovery (R1),"929,597"
14,Makroregion południowy,Rest capacity - cubic metres,"Disposal - landfill and other (D1-D7, D12)","298,514"
0,Makroregion południowy,Capacity - tonnes per year,Disposal - incineration (D10),"197,973"
1,Makroregion południowy,Capacity - tonnes per year,"Disposal - landfill and other (D1-D7, D12)","31,034"
2,Makroregion południowy,Capacity - tonnes per year,"Disposal - other (D2-D4, D6-D7)","31,034"
12,Makroregion południowy,Facilities - number,Recovery - recycling and backfilling (R2-R11),461.70
11,Makroregion południowy,Facilities - number,Recovery - recycling,387.60
6,Makroregion południowy,Facilities - number,"Disposal - landfill (D1, D5, D12)",62.90


## Waste shipment data

- Shows if the generated waste is being exported.
- Shows if the country is an importer or exporter of that type of waste.
    - I.e has capacity to treat that waste, and whether they use disposal or recovery operations.

Detail of the waste shipment data is very high, up to 6-digit LoW codes. 
- **See repository data/processed/Waste shipment for more detail!**

Example: Poland, Metallic wastes 

In [119]:
wasship = pd.read_csv('data/interim/wasship_pivoted.csv',sep=';')

poland_metals = wasship[(wasship['Top_Level_Description']=='Metallic wastes')&
        (wasship['Country reporting']=='Poland')].sort_values(by='mean_ship',ascending=False)


poland_metals.groupby(['Import/export','Disposal and recovery code'])['mean_ship'].sum().reset_index().sort_values(by='mean_ship',ascending=False)

,Import/export,Disposal and recovery code,mean_ship
0,Import,R4,35.03


No metallic wastes being imported/exported to Poland as legal 'waste'. 

- But about 6 M tonnes as secondary raw materials!





### Conclusions:

- Poland generates annually a large amount of metallic wastes (2-3 million)
    - It exports ~2 million of metallic wastes as secondary raw materials, mostly to Intra-EU
    - Imports ~1 million metallic wastes as secondary raw materials 

- No export or import of metallic wastes as 'legal waste shipments'
- Poland registers ~8 million of treated metallic wastes (as recovery/recycling)
    - This could e.g. be old storage facilities of metallic wastes that's getting recovered.




-> Although this was a very quick display, it shows how you can use these Eurostat datasets to track waste sources and find hotspots

Other available datasets:

- MFA data.
- Material import dependency per country and material groups -> shows which countries are dependent on importing raw materials.

...and more




In [140]:
mfa = extend_eurostat_dataset(load_dataset('env_ac_mfa'),['indic_env','material','unit','geo'])

MFA data could be very useful as well... 

- Showing Poland's DMC of top raw materials

... to be continued

In [159]:
poland_industrial_mf = mfa[(mfa['indic_env_description']=='Domestic material consumption')&
    (mfa['geo']=='PL')&
    (mfa['unit_description']=='Thousand tonnes')&
    mfa['material_description'].isin([ 'Metal ores (gross ores)', 'Iron',
       'Non-ferrous metal', 'Copper', 'Nickel', 'Lead', 'Zinc', 'Tin',
       'Gold, silver, platinum and other precious metals',
       'Bauxite and other aluminium', 'Uranium and thorium',
       'Other non-ferrous metals', 'Products mainly from metals',
       'Non-metallic minerals',
       'Marble, granite, sandstone, porphyry, basalt, other ornamental or building stone (excluding slate)',
       'Chalk and dolomite', 'Slate', 'Chemical and fertiliser minerals',
       'Salt', 'Limestone and gypsum', 'Clays and kaolin',
       'Sand and gravel', 'Other non-metallic minerals n.e.c.',
       'Products mainly from non metallic minerals',
       'Fossil energy materials/carriers',
       'Coal and other solid energy materials/carriers',
       'Lignite (brown coal)', 'Hard coal', 'Oil shale and tar sands'])]

poland_industrial_mf.groupby('material_description')[poland_industrial_mf.columns[19:]].sum().sort_values(by='2023',ascending=False).head(15)



,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
material_description,,,,,,,,,,,,,,,,,,,,,,,,,
Non-metallic minerals,"171,227","146,162","137,291","152,610","162,618","182,636","209,013","254,755","270,042","255,907","292,866","417,572","317,390","291,471","274,442","295,058","297,389","323,582","351,105","336,229","328,489","337,571","323,258","311,650","299,103"
Sand and gravel,"89,972","75,343","69,057","81,878","83,958","102,109","119,760","145,929","157,063","146,979","169,601","259,265","191,543","176,160","149,572","171,307","176,551","189,690","201,439","188,251","184,997","189,383","174,467","169,787",0.00
Fossil energy materials/carriers,"167,039","168,216","168,267","173,154","173,855","174,545","168,661","165,974","172,759","165,391","163,436","173,004","172,238","164,852","160,934","159,405","157,613","163,576","169,972","157,421","141,022","146,584","156,742","136,320","125,093"
Coal and other solid energy materials/carriers,"140,449","142,239","141,594","145,598","145,200","143,729","144,835","140,043","146,427","138,243","137,367","147,807","147,685","142,989","138,844","135,909","131,237","133,673","137,844","125,662","110,142","114,911","123,931","102,296",0.00
"Marble, granite, sandstone, porphyry, basalt, other ornamental or building stone (excluding slate)","24,070","22,096","22,737","26,047","28,864","33,906","36,725","45,887","51,371","55,603","63,581","85,856","65,371","58,605","64,414","64,609","59,958","71,106","81,459","78,956","76,749","79,257","79,903","79,710",0.00
Hard coal,"80,476","82,151","82,865","84,091","83,163","81,111","82,937","81,430","85,539","79,902","79,771","83,638","81,989","75,811","73,655","71,265","69,600","71,257","77,725","73,901","62,568","60,949","67,525","60,521",0.00
Limestone and gypsum,"36,097","30,030","27,595","27,544","30,036","29,436","35,383","40,511","39,875","36,464","41,224","49,992","42,083","39,739","42,007","43,211","42,424","43,343","46,655","47,320","47,347","48,028","49,902","45,399",0.00
Lignite (brown coal),"59,471","59,539","58,170","60,886","61,172","61,630","60,853","57,549","59,695","57,077","56,444","62,785","64,333","65,863","63,820","63,248","60,345","61,259","58,604","50,418","46,087","52,474","54,891","40,350",0.00
Metal ores (gross ores),"38,868","38,812","37,765","36,436","38,757","35,398","39,319","37,804","35,617","28,928","31,413","31,136","37,806","38,640","40,666","41,252","42,458","41,283","41,863","34,938","37,216","38,562","37,647","36,081","37,804"


In [149]:
mfa['indic_env_description'].unique()



array(['Domestic extraction', 'Domestic material consumption',
       'Direct material inputs', 'Exports', 'Imports',
       'Physical trade balance'], dtype=object)

### WEEE datasets

In [17]:
env_waselee = load_dataset("env_waselee")  # WEEE by management operations
env_waseleeos = load_dataset(
    "env_waseleeos"
)  # WEEE by management operations, open scope

In [25]:
#env_waselee = extend_eurostat_dataset(load_dataset("env_waselee"),['waste','wst_oper','geo'])
env_waselee

,freq,waste,waste_description,wst_oper,wst_oper_description,unit,geo,geo_description,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
0,A,EE_ATD,Automatic dispensers,COL,Waste collected,KG_HAB,AT,Austria,0.00,0.01,0.01,0.02,0.00,0.01,0.01,0.01,0.01,0.01,0.01,0.02,0.03,0.01
1,A,EE_ATD,Automatic dispensers,COL,Waste collected,KG_HAB,BE,Belgium,NaN,NaN,0.00,0.03,0.19,0.12,0.15,0.17,0.21,0.24,0.06,0.07,0.04,0.25
2,A,EE_ATD,Automatic dispensers,COL,Waste collected,KG_HAB,BG,Bulgaria,NaN,NaN,NaN,NaN,0.01,0.03,0.03,0.04,0.02,0.00,0.02,0.02,0.03,NaN
3,A,EE_ATD,Automatic dispensers,COL,Waste collected,KG_HAB,CY,Cyprus,NaN,NaN,0.00,0.00,0.00,0.00,0.01,0.02,0.01,0.01,0.08,0.00,0.00,0.05
4,A,EE_ATD,Automatic dispensers,COL,Waste collected,KG_HAB,CZ,Czechia,NaN,NaN,0.00,0.00,0.00,0.01,0.01,0.01,0.00,0.00,0.02,0.03,0.01,0.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10012,A,TOTAL,Total waste,TRT_NEU,Waste treated outside the EU,T,RO,Romania,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
10013,A,TOTAL,Total waste,TRT_NEU,Waste treated outside the EU,T,SE,Sweden,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
10014,A,TOTAL,Total waste,TRT_NEU,Waste treated outside the EU,T,SI,Slovenia,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
10015,A,TOTAL,Total waste,TRT_NEU,Waste treated outside the EU,T,SK,Slovakia,0.00,0.00,0.00,0.00,0.00,6.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [26]:
env_waselee['waste_description'].unique()

array(['Automatic dispensers',
       'Consumer equipment and photovoltaic panels', 'Consumer equipment',
       'Photovoltaic panels', 'Electrical and electronic tools',
       'IT and telecommunications equipment',
       'Large household appliances', 'Lighting equipment',
       'Gas discharge lamps', 'Medical devices',
       'Monitoring and control instruments', 'Small household appliances',
       'Toys, leisure and sports equipment', 'Total waste'], dtype=object)

In [30]:
env_waselee['wst_oper_description'].unique()

array(['Waste collected', 'Waste collected from households',
       'Waste collected from other sources', 'Products put on the market',
       'Preparing for reuse', 'Recovery',
       'Recycling and preparing for reuse', 'Waste treatment',
       'Waste treated in another Member State of the EU',
       'Waste treated in the Member State',
       'Waste treated outside the EU'], dtype=object)

In [29]:
env_waseleeos = extend_eurostat_dataset(load_dataset("env_waseleeos"),['waste','wst_oper','geo'])
env_waseleeos['waste_description'].unique()

array(['Waste arising only from separate collection of EEE (6 categories methodology defined in WEEE directive)',
       'Large equipment (any external dimension more than 50 cm)',
       'Large equipment excluding photovoltaic panels',
       'Photovoltaic panels', 'Lamps',
       'Small equipment (no external dimension more than 50 cm)',
       'Small IT and telecommunications equipment (no external dimension more than 50 cm)',
       'Screens, monitors, and equipment containing screens having a surface greater than 100 cm2',
       'Temperature exchange equipment'], dtype=object)